# 04 - Preprocessing and Training

**Objectif :** préparer les matrices sans fuite de données, entraîner les candidats et sélectionner le meilleur modèle.

In [ ]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

## 1. Load training partition

In [ ]:
from credit_risk_lab.infrastructure.data_sources import CsvLoanDataLoader

train_loader = CsvLoanDataLoader(path=settings.train_path)
train_raw = train_loader.load()
train_raw.head()

## 2. Feature engineering

In [ ]:
from credit_risk_lab.infrastructure.feature_engineering import LoanFeatureEngineer

feature_engineer = LoanFeatureEngineer()
train_features = feature_engineer.transform(train_raw)
train_features.shape

## 3. Development split

In [ ]:
from credit_risk_lab.application import three_way_stratified_split

split = three_way_stratified_split(train_features)

pd.DataFrame(
    [
        {"split": "train", "rows": len(split.y_train), "positive_rate": split.y_train.mean()},
        {"split": "validation", "rows": len(split.y_validation), "positive_rate": split.y_validation.mean()},
        {"split": "test", "rows": len(split.y_test), "positive_rate": split.y_test.mean()},
    ]
)

## 4. Sensitive columns excluded from training features

In [ ]:
sensitive_columns = [
    column for column in settings.sensitive_columns if column in split.x_train
]

x_train = split.x_train.drop(columns=sensitive_columns)
x_validation = split.x_validation.drop(columns=sensitive_columns)
x_test = split.x_test.drop(columns=sensitive_columns)
sensitive_test = split.x_test[sensitive_columns].copy()

{"excluded": sensitive_columns, "training_columns": x_train.shape[1]}

## 5. Fit preprocessing on train only

In [ ]:
from credit_risk_lab.infrastructure.modeling import CreditRiskPreprocessor

preprocessor = CreditRiskPreprocessor()
x_train_t = preprocessor.fit_transform(x_train)
x_validation_t = preprocessor.transform(x_validation)
x_test_t = preprocessor.transform(x_test)

{
    "train_matrix": x_train_t.shape,
    "validation_matrix": x_validation_t.shape,
    "test_matrix": x_test_t.shape,
    "first_output_features": preprocessor.feature_names_[:20],
}

## 6. Inspect configured model candidates

In [ ]:
from credit_risk_lab.infrastructure.modeling import build_configured_models, load_models_config

model_config = load_models_config()
configured_models = build_configured_models(
    random_state=settings.random_state,
    config=model_config,
)

pd.DataFrame(
    [
        {
            "model": model.name,
            "parameters": model.parameters,
            "early_stopping": model.early_stopping_rounds,
        }
        for model in configured_models
    ]
)

## 7. Train candidates

In [ ]:
from credit_risk_lab.infrastructure.modeling import BoostingModelTrainer

trainer = BoostingModelTrainer(random_state=settings.random_state)
training_results = trainer.fit(
    x_train_t,
    split.y_train,
    x_validation_t,
    split.y_validation,
)

validation_metrics = trainer.results_frame(training_results)
validation_metrics.round(4)

## 8. Select best model

In [ ]:
from credit_risk_lab.infrastructure.modeling import BestModelSelector

selector = BestModelSelector(metric=settings.selection_metric)
best_result = selector.select(training_results)

{
    "selected_model": best_result.model_name,
    "selection_metric": settings.selection_metric,
    "threshold": best_result.threshold,
}

## 9. Training curves

In [ ]:
from credit_risk_lab.infrastructure.visualization import plot_learning_curves, plot_model_comparison

histories = {
    result.model_name: result.history for result in training_results
}

plot_model_comparison(validation_metrics).show()
plot_learning_curves(histories).show()